<a href="https://colab.research.google.com/github/thahsinj06/Fly-rank-ml-internship-work/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thahsinj06/Fly-rank-ml-internship-work/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

##1. Method choice and why
Logistic Regression was selected as the first modelling method because the target is binary and the model provides an interpretable benchmark against the Week-4 rule-based baseline. The model uses signals examined during the audit: search volume, staleness, average position, and CTR. CTR is included as a cautious feature because the audit found a mixed relationship, while its skew is handled through transformation. The goal is decision-support rather than maximising complexity.

In [4]:
# ============================================
# SECTION 1 — METHOD CHOICE
# Logistic Regression
# ============================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# --------------------------------------------
# 1. Load dataset
# --------------------------------------------

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "thahsinj06/Fly-rank-ml-internship-work/"
    "main/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_URL)

print("Dataset shape:", df.shape)

# --------------------------------------------
# 2. Create target
# --------------------------------------------
# 1 = declining
# 0 = not declining

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print("\nTarget distribution:")
print(df["is_declining_label"].value_counts())

print("\nTarget percentage:")
print(
    (df["is_declining_label"].value_counts(normalize=True) * 100)
    .round(2)
)

# --------------------------------------------
# 3. Select features
# --------------------------------------------
# Do NOT include trend_direction or trend_pct
# because they are used to create the target.

FEATURES = [
    "search_volume",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr"
]

TARGET = "is_declining_label"

X = df[FEATURES].copy()
y = df[TARGET].copy()

# --------------------------------------------
# 4. Handle missing/infinite values
# --------------------------------------------

X = X.replace([np.inf, -np.inf], np.nan)

X = X.fillna(X.median(numeric_only=True))

# --------------------------------------------
# 5. Create Logistic Regression pipeline
# --------------------------------------------

logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

print("\nFeatures used:")
print(FEATURES)

print("\nTarget:", TARGET)

print("\nLogistic Regression pipeline created successfully.")

Dataset shape: (30000, 44)

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Target percentage:
is_declining_label
1    54.21
0    45.79
Name: proportion, dtype: float64

Features used:
['search_volume', 'impressions_90d', 'days_since_last_update', 'avg_position', 'ctr']

Target: is_declining_label

Logistic Regression pipeline created successfully.


## 2. Split design

A stratified 80/20 train-test split is used so both declining and non-declining pages are represented in similar proportions in each set. The test set is held out until final evaluation. This provides a simple, reproducible validation design for the binary classification question.

In [5]:
# ============================================
# SECTION 2 — SPLIT DESIGN
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True).round(3))

Training samples: 24000
Test samples: 6000

Training target distribution:
is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64

Test target distribution:
is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64


## 3. Train + compare vs my baseline

Logistic Regression is trained using the audited signals. The Week-4 rule is converted into a binary prediction where REFRESH and REVIEW represent higher-priority pages and MONITOR represents lower priority. Both approaches are evaluated on the same held-out test rows using the same classification metrics.

In [6]:
# ============================================
# SECTION 3 — TRAIN + COMPARE
# ============================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# --------------------------------------------
# 1. Train Logistic Regression
# --------------------------------------------

logistic_model.fit(X_train, y_train)

# Generate predictions
y_pred = logistic_model.predict(X_test)

# --------------------------------------------
# 2. Evaluate Logistic Regression
# --------------------------------------------

model_results = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred)
}

# --------------------------------------------
# 3. Recreate Week-4 baseline on test set
# --------------------------------------------

baseline_test = df.loc[X_test.index].copy()

# Volume score
baseline_test["_volume_score"] = (
    baseline_test["impressions_90d"].rank(pct=True)
)

# Staleness score
baseline_test["_staleness_score"] = (
    baseline_test["days_since_last_update"]
    .clip(upper=180) / 180
)

# Week-4 baseline score
baseline_test["_baseline_score"] = (
    0.60 * baseline_test["_volume_score"]
    + 0.40 * baseline_test["_staleness_score"]
)

# Week-4 actions
baseline_test["_action"] = np.select(
    [
        baseline_test["_baseline_score"] >= 0.70,
        baseline_test["_baseline_score"] >= 0.45
    ],
    [
        "REFRESH",
        "REVIEW"
    ],
    default="MONITOR"
)

# --------------------------------------------
# 4. Convert baseline actions to binary
# --------------------------------------------
# REFRESH + REVIEW = potential decline
# MONITOR = not flagged

baseline_pred = (
    baseline_test["_action"]
    .isin(["REFRESH", "REVIEW"])
    .astype(int)
)

# --------------------------------------------
# 5. Evaluate Week-4 baseline
# --------------------------------------------

baseline_results = {
    "Accuracy": accuracy_score(y_test, baseline_pred),
    "Precision": precision_score(y_test, baseline_pred),
    "Recall": recall_score(y_test, baseline_pred),
    "F1": f1_score(y_test, baseline_pred)
}

# --------------------------------------------
# 6. Comparison table
# --------------------------------------------

comparison = pd.DataFrame(
    {
        "Week-4 Baseline": baseline_results,
        "Logistic Regression": model_results
    }
)

print("Model comparison:")
display(comparison.round(3))

Model comparison:


,Week-4 Baseline,Logistic Regression
Accuracy,0.541,0.557
Precision,0.596,0.558
Recall,0.475,0.873
F1,0.529,0.681


## 4. Errors and interpretation

False positives are pages predicted as declining that are not labelled declining, while false negatives are declining pages missed by the model. Feature coefficients are inspected to understand which signals the model leans on. These results are treated as observed associations and decision-support, not causal evidence.

In [7]:
# ============================================
# SECTION 4 — ERRORS AND INTERPRETATION
# ============================================

from sklearn.metrics import confusion_matrix

# --------------------------------------------
# 1. Confusion matrix
# --------------------------------------------

cm = confusion_matrix(y_test, y_pred)

tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix:")
print(cm)

print("\nError summary:")
print("True Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives  :", tp)

# --------------------------------------------
# 2. Logistic Regression coefficients
# --------------------------------------------

coefficients = pd.DataFrame({
    "Feature": FEATURES,
    "Coefficient": logistic_model.named_steps["model"].coef_[0]
})

coefficients["Absolute Effect"] = (
    coefficients["Coefficient"].abs()
)

coefficients = coefficients.sort_values(
    "Absolute Effect",
    ascending=False
)

print("\nFeature influence:")
display(coefficients.round(4))

# --------------------------------------------
# 3. Simple interpretation
# --------------------------------------------

print("\nInterpretation:")

for _, row in coefficients.iterrows():
    direction = "positive" if row["Coefficient"] > 0 else "negative"
    print(
        f"{row['Feature']}: {direction} association "
        f"with predicted decline "
        f"(coefficient = {row['Coefficient']:.4f})"
    )

Confusion Matrix:
[[ 503 2245]
 [ 414 2838]]

Error summary:
True Negatives : 503
False Positives: 2245
False Negatives: 414
True Positives  : 2838

Feature influence:


,Feature,Coefficient,Absolute Effect
4,ctr,-0.1883,0.1883
2,days_since_last_update,0.1859,0.1859
3,avg_position,-0.0927,0.0927
1,impressions_90d,-0.0756,0.0756
0,search_volume,-0.0256,0.0256



Interpretation:
ctr: negative association with predicted decline (coefficient = -0.1883)
days_since_last_update: positive association with predicted decline (coefficient = 0.1859)
avg_position: negative association with predicted decline (coefficient = -0.0927)
impressions_90d: negative association with predicted decline (coefficient = -0.0756)
search_volume: negative association with predicted decline (coefficient = -0.0256)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.